# 补充评测 Notebook

**目标**：补跑缺失的评测数据，统一使用自定义 eval 脚本（不用 lm-eval）。

| Phase | 内容 | 预计耗时 |
|-------|------|----------|
| **E1** | 1.5B Baseline（GSM8K + MATH-500，n=200）| ~15 min |
| **E2** | 7B Baseline via API（GSM8K + MATH-500，n=200）| ~20 min |
| **E3** | Group A DPO 评测（GSM8K + MATH-500，n=200）| ~20 min |
| **E5** | Group C Teacher DPO 数据补全（用 SFT badcase 填充 rejected）| ~30 min |
| **E6** | Group C DPO 训练（DoRA + 五段课程 SFT + Teacher DPO）| ~20 min |
| **E7** | Group C 评测（GSM8K + MATH-500，n=200）| ~20 min |
| **E4** | 汇总表 | ~1 min |

**评测方式**：`eval/gsm8k_eval.py` + `eval/math_eval.py`（自定义，与之前 n=200 的实验一致）。
**输出**：`logs/eval_supplement_*.json`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E0: 环境准备
# ═══════════════════════════════════════════════════════════════════
!pip install -q -U pip
!pip install -q "unsloth>=2025.1.0" "trl>=0.14.0" "peft>=0.14.0" "bitsandbytes>=0.45.0" \
    "transformers>=4.49.0" "datasets>=3.2.0" "accelerate>=1.2.0" \
    "pyyaml>=6.0.2" "safetensors" "tqdm" "scipy" "sympy" "requests"

import torch, os, json, subprocess, sys
from pathlib import Path
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} | VRAM: {gpu.total_memory / 1e9:.1f} GB')

# 全局常量
EVAL_N = '200'
BIT = ['--load_in_4bit']

def run_eval(cmd, label):
    """运行评测子进程，实时打印输出。"""
    cmd = [cmd[0], '-u'] + cmd[1:]
    print(f'  执行: {" ".join(cmd[-8:])}')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        print(f'  ❌ {label} 失败 (exit {proc.returncode})')
        return False
    return True

def print_result(path, label):
    """打印评测结果摘要。"""
    if not os.path.isfile(path):
        print(f'  {label}: 结果文件不存在 ({path})')
        return
    d = json.load(open(path))
    acc = d.get('accuracy', d.get('macro_avg_accuracy', 'N/A'))
    total = d.get('total', 'N/A')
    if isinstance(acc, float):
        print(f'  {label}: {acc:.1%} ({total}题)')
    else:
        print(f'  {label}: {acc} (n={total})')

In [ ]:
# ── 0.2 挂载 Drive + 设置路径 ─────────────────────────────────────
from google.colab import drive, userdata

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/Qwen-Reasoning'
os.chdir(PROJECT_DIR)
print(f'工作目录: {PROJECT_DIR}')

# 设置 API key（7B 评测需要）
try:
    os.environ['DASHSCOPE_API_KEY'] = userdata.get('DASHSCOPE_API_KEY')
    print('DASHSCOPE_API_KEY: OK')
except Exception:
    print('DASHSCOPE_API_KEY: missing（7B 评测将跳过）')

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    print('HF_TOKEN: OK')
except Exception:
    print('HF_TOKEN: missing')

sys.path.insert(0, f'{PROJECT_DIR}/eval')
sys.path.insert(0, f'{PROJECT_DIR}/scripts')
for d in ['logs', 'outputs']:
    os.makedirs(d, exist_ok=True)

In [ ]:
# ── 0.3 检查模型 + NF4 检测 ─────────────────────────────────────
from model_loader import _has_quantized_weights, _find_adapter_dir

def ensure_fp16_merged(merged_path, adapter_path_hint, label):
    """检测 NF4 并自动重合并为 fp16，返回可用路径。"""
    if not merged_path or not os.path.isfile(f'{merged_path}/config.json'):
        return merged_path
    if not _has_quantized_weights(merged_path):
        return merged_path
    adapters = _find_adapter_dir(merged_path)
    if not adapters:
        if adapter_path_hint and os.path.isfile(f'{adapter_path_hint}/adapter_config.json'):
            adapters = [adapter_path_hint]
        else:
            print(f'  ❌ {label}: NF4 且未找到 adapter')
            return merged_path
    adapter = adapters[0] if isinstance(adapters, list) else adapters
    fp16_path = merged_path.rstrip('/') + '_fp16'
    if os.path.isfile(f'{fp16_path}/config.json'):
        print(f'  {label}: fp16 版已存在 → {fp16_path}')
        return fp16_path
    print(f'  {label}: NF4 → 重合并 fp16: {adapter} → {fp16_path}')
    subprocess.run(['python3', 'scripts/merge_lora.py',
                    '--adapter_path', adapter, '--output_path', fp16_path], check=True)
    print(f'  ✅ {label} fp16 完成: {fp16_path}')
    return fp16_path

# 检查各模型
BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
G_A_DPO = 'outputs/group_a/dpo'
G_A_MERGED = 'outputs/group_a/merged'

# Group A DPO: 检查 adapter 和 merged
if os.path.isfile(f'{G_A_MERGED}/config.json'):
    print(f'Group A merged: OK ({G_A_MERGED})')
    G_A_MERGED = ensure_fp16_merged(G_A_MERGED, G_A_DPO, 'Group A DPO')
elif os.path.isfile(f'{G_A_DPO}/adapter_config.json'):
    print(f'Group A adapter 存在但 merged 不存在，需要先合并')
    print(f'  执行 merge_lora.py ...')
    os.makedirs(G_A_MERGED, exist_ok=True)
    # Group A DPO 的 base 是 Group A SFT merged
    g_a_sft_merged = 'outputs/group_a/sft_merged'
    subprocess.run([
        'python3', 'scripts/merge_lora.py',
        '--adapter_path', G_A_DPO,
        '--base_model', g_a_sft_merged,
        '--output_path', G_A_MERGED,
    ], check=True)
    print(f'  ✅ Group A DPO 合并完成: {G_A_MERGED}')
    G_A_MERGED = ensure_fp16_merged(G_A_MERGED, G_A_DPO, 'Group A DPO')
else:
    print(f'⚠️ Group A DPO 模型不存在（{G_A_DPO} 和 {G_A_MERGED}）')
    print(f'  请确认 Drive 上有 outputs/group_a/dpo/ 或 outputs/group_a/merged/')

# 检查已有结果
print('\n📋 已有结果:')
for f in ['logs/eval_supplement_1.5b_gsm8k.json', 'logs/eval_supplement_1.5b_math.json',
          'logs/eval_supplement_7b_gsm8k.json', 'logs/eval_supplement_7b_math.json',
          'logs/eval_supplement_group_a_dpo_gsm8k.json', 'logs/eval_supplement_group_a_dpo_math.json']:
    if os.path.isfile(f):
        d = json.load(open(f))
        acc = d.get('accuracy', 'N/A')
        total = d.get('total', 'N/A')
        print(f'  ✅ {f}: acc={acc}, n={total}')
    else:
        print(f'  ❌ {f}: 未跑')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E1: 1.5B Baseline 评测（GSM8K + MATH-500，n=200）
# 模型：Qwen/Qwen2.5-1.5B-Instruct（未经任何微调）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

print('\n' + '='*60)
print('  E1: 1.5B Baseline 评测')
print('='*60)

# GSM8K
gsm_out = 'logs/eval_supplement_1.5b_gsm8k.json'
if not os.path.isfile(gsm_out):
    run_eval([
        'python3', 'eval/gsm8k_eval.py',
        '--model_path', BASE_MODEL,
        '--max_samples', EVAL_N,
        '--output', gsm_out,
    ] + BIT, '1.5B GSM8K')
print_result(gsm_out, '1.5B GSM8K')

# MATH-500
math_out = 'logs/eval_supplement_1.5b_math.json'
if not os.path.isfile(math_out):
    run_eval([
        'python3', 'eval/math_eval.py',
        '--model_path', BASE_MODEL,
        '--max_samples', EVAL_N,
        '--output', math_out,
    ] + BIT, '1.5B MATH')
print_result(math_out, '1.5B MATH')

print('\nE1 完成')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E2: 7B Baseline 评测（DashScope API，GSM8K + MATH-500，n=200）
# 模型：qwen2.5-7b-instruct（通过 API 调用）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

api_key = os.environ.get('DASHSCOPE_API_KEY', '')
if not api_key:
    print('⚠️ DASHSCOPE_API_KEY 未设置，跳过 E2（7B 评测）')
    print('  请在 Colab Secrets 中设置 DASHSCOPE_API_KEY')
else:
    print('\n' + '='*60)
    print('  E2: 7B Baseline 评测 (DashScope API)')
    print('='*60)

    DASHSCOPE_URL = 'https://dashscope.aliyuncs.com/compatible-mode/v1'

    # 7B GSM8K (API)
    gsm7b_out = 'logs/eval_supplement_7b_gsm8k.json'
    if not os.path.isfile(gsm7b_out):
        run_eval([
            'python3', '-u', 'eval/gsm8k_api_eval.py',
            '--api_base_url', DASHSCOPE_URL,
            '--api_key', api_key,
            '--model', 'qwen2.5-7b-instruct',
            '--max_samples', EVAL_N,
            '--output', gsm7b_out,
        ], '7B GSM8K')
    print_result(gsm7b_out, '7B GSM8K')

    # 7B MATH-500 (API)
    math7b_out = 'logs/eval_supplement_7b_math.json'
    if not os.path.isfile(math7b_out):
        run_eval([
            'python3', '-u', 'eval/math_api_eval.py',
            '--api_base_url', DASHSCOPE_URL,
            '--api_key', api_key,
            '--model', 'qwen2.5-7b-instruct',
            '--max_samples', EVAL_N,
            '--output', math7b_out,
        ], '7B MATH')
    print_result(math7b_out, '7B MATH')

    print('\nE2 完成')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E3: Group A DPO 评测（GSM8K + MATH-500，n=200）
# 模型：outputs/group_a/merged（LoRA + 单段SFT + Standard DPO）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

# 确保 G_A_MERGED 变量存在且模型可用
if 'G_A_MERGED' not in dir() or not G_A_MERGED:
    G_A_MERGED = 'outputs/group_a/merged'

if not os.path.isfile(f'{G_A_MERGED}/config.json'):
    print('⚠️ Group A DPO 模型不存在，跳过 E3')
    print(f'  检查: {G_A_MERGED}/config.json')
else:
    print('\n' + '='*60)
    print(f'  E3: Group A DPO 评测 (模型: {G_A_MERGED})')
    print('='*60)

    # GSM8K
    gsm_ga_out = 'logs/eval_supplement_group_a_dpo_gsm8k.json'
    if not os.path.isfile(gsm_ga_out):
        run_eval([
            'python3', 'eval/gsm8k_eval.py',
            '--model_path', G_A_MERGED,
            '--max_samples', EVAL_N,
            '--output', gsm_ga_out,
        ] + BIT, 'Group A DPO GSM8K')
    print_result(gsm_ga_out, 'Group A DPO GSM8K')

    # MATH-500
    math_ga_out = 'logs/eval_supplement_group_a_dpo_math.json'
    if not os.path.isfile(math_ga_out):
        run_eval([
            'python3', 'eval/math_eval.py',
            '--model_path', G_A_MERGED,
            '--max_samples', EVAL_N,
            '--output', math_ga_out,
        ] + BIT, 'Group A DPO MATH')
    print_result(math_ga_out, 'Group A DPO MATH')

    print('\nE3 完成')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E5: Group C Teacher DPO 数据补全
# 用 SFT badcase 填充 rejected 字段（已有 1500 chosen，缺 rejected）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

print('\n' + '='*60)
print('  E5: Group C Teacher DPO 数据补全')
print('='*60)

# 检查 badcase 文件
badcase_candidates = [
    'logs 2/gsm8k_sft_badcases.jsonl',
    'logs/gsm8k_sft_badcases.jsonl',
]
BADCASE_FILE = None
for bp in badcase_candidates:
    if os.path.isfile(bp):
        BADCASE_FILE = bp
        break

if not BADCASE_FILE:
    print('⚠️ 未找到 SFT badcase 文件，尝试从 SFT 评测结果生成...')
    sft_gsm_out = 'logs/eval_supplement_sft_gsm8k.json'
    sft_gsm_badcase = 'logs/gsm8k_sft_badcases.jsonl'
    if not os.path.isfile(sft_gsm_badcase):
        sft_model = 'outputs/sft_merged'
        if os.path.isfile(f'{sft_model}/config.json'):
            run_eval([
                'python3', 'eval/gsm8k_eval.py',
                '--model_path', sft_model,
                '--max_samples', '300',
                '--output', sft_gsm_out,
                '--badcase_output', sft_gsm_badcase,
            ] + BIT, 'SFT GSM8K (for badcases)')
            BADCASE_FILE = sft_gsm_badcase
        else:
            print('❌ SFT merged 模型不存在，无法生成 badcase')
            print('  请确保 outputs/sft_merged/ 或 logs 2/gsm8k_sft_badcases.jsonl 存在')

if BADCASE_FILE:
    n_bad = sum(1 for _ in open(BADCASE_FILE))
    print(f'  Badcase 文件: {BADCASE_FILE} ({n_bad} 条)')

    # 检查现有 teacher DPO 数据
    teacher_data = 'data/processed/dpo_teacher_round_1.json'
    if os.path.isfile(teacher_data):
        existing = json.load(open(teacher_data))
        n_chosen = sum(1 for d in existing if d.get('chosen'))
        n_rejected = sum(1 for d in existing if d.get('rejected'))
        print(f'  现有 teacher 数据: {len(existing)} 条 (chosen={n_chosen}, rejected={n_rejected})')
    else:
        print(f'  现有 teacher 数据不存在，将从头生成')

    # 运行 build_teacher_dpo.py 补全 rejected
    teacher_out = 'data/processed/dpo_teacher_round_1_complete.json'
    if not os.path.isfile(teacher_out):
        print(f'\n  开始补全 teacher DPO 数据...')
        print(f'  使用 badcase: {BADCASE_FILE}')
        run_eval([
            'python3', '-u', 'gpu/scripts/build_teacher_dpo.py',
            '--rejected_jsonl', BADCASE_FILE,
            '--output', teacher_out,
            '--max_samples', '1500',
            '--workers', '4',
        ], 'Teacher DPO 数据补全')

    if os.path.isfile(teacher_out):
        data = json.load(open(teacher_out))
        n_ok = sum(1 for d in data if d.get('chosen') and d.get('rejected'))
        print(f'  ✅ 补全完成: {len(data)} 条, 有效 (chosen+rejected) = {n_ok} 条')
    else:
        print(f'  ❌ 补全失败，输出文件不存在: {teacher_out}')
        teacher_out = teacher_data
        print(f'  将使用原始数据: {teacher_out}')

    print('\nE5 完成')
else:
    print('❌ 无法继续: 无 badcase 文件且无法生成')
    teacher_out = 'data/processed/dpo_teacher_round_1.json'

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E6: Group C DPO 训练（DoRA + 五段课程 SFT + Teacher DPO）
# Base: outputs/sft_merged（与 Group B 相同的 SFT 基座）
# Data: data/processed/dpo_teacher_round_1_complete.json（E5 产出）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

print('\n' + '='*60)
print('  E6: Group C DPO 训练')
print('='*60)

# 路径定义
G_C_DPO = 'outputs/group_c/dpo'
G_C_MERGED = 'outputs/group_c/merged'
SFT_BASE = 'outputs/sft_merged'

# 确定 teacher 数据路径（E5 产出优先，fallback 到原始）
teacher_data_path = 'data/processed/dpo_teacher_round_1_complete.json'
if not os.path.isfile(teacher_data_path):
    teacher_data_path = 'data/processed/dpo_teacher_round_1.json'
    print(f'  ⚠️ 使用原始 teacher 数据（无 rejected）: {teacher_data_path}')
else:
    print(f'  使用补全后的 teacher 数据: {teacher_data_path}')

# 检查 SFT base
if not os.path.isfile(f'{SFT_BASE}/config.json'):
    print(f'❌ SFT base 模型不存在: {SFT_BASE}/config.json')
    print(f'  Group C 训练需要 DoRA+五段课程 SFT 基座')
else:
    # 检查是否已训练完成
    if os.path.isfile(f'{G_C_MERGED}/config.json'):
        print(f'✅ Group C merged 模型已存在: {G_C_MERGED}')
    elif os.path.isfile(f'{G_C_DPO}/adapter_config.json'):
        print(f'✅ Group C DPO adapter 已存在: {G_C_DPO}')
        print(f'  需要合并为 merged 模型...')
    else:
        # 写入 Group C 专用 DPO 配置
        g_c_config = 'config/dpo_group_c.yaml'
        import yaml
        gc_cfg = {
            'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
            'base_adapter_path': SFT_BASE,
            'output_dir': G_C_DPO,
            'max_seq_length': 2048,
            'load_in_4bit': True,
            'seed': 42,
            'beta': 0.1,
            'loss_type': 'sigmoid',
            'dataset': {
                'name': 'local',
                'split': 'train',
                'max_samples': 1500,
            },
            'lora': {
                'use_dora': True,
                'r': 16,
                'alpha': 32,
                'dropout': 0.0,
                'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                   'gate_proj', 'up_proj', 'down_proj'],
            },
            'train': {
                'per_device_train_batch_size': 1,
                'gradient_accumulation_steps': 16,
                'warmup_steps': 50,
                'max_steps': 600,
                'learning_rate': 1e-5,
                'logging_steps': 10,
                'save_steps': 100,
                'eval_steps': 100,
                'weight_decay': 0.0,
                'lr_scheduler_type': 'cosine',
                'optim': 'paged_adamw_8bit',
                'fp16': False,
                'bf16': True,
                'dataloader_num_workers': 4,
                'dataloader_pin_memory': True,
            },
            'dataset_path': teacher_data_path,
        }
        with open(g_c_config, 'w') as f:
            yaml.dump(gc_cfg, f, default_flow_style=False, allow_unicode=True)
        print(f'  配置已写入: {g_c_config}')
        print(f'  数据: {teacher_data_path}')
        print(f'  输出: {G_C_DPO}')

        # 运行 DPO 训练
        run_eval([
            'python3', '-u', 'scripts/dpo_train.py',
            '--config', g_c_config,
        ], 'Group C DPO 训练')

    # 合并 LoRA → merged 模型
    if os.path.isfile(f'{G_C_DPO}/adapter_config.json') and not os.path.isfile(f'{G_C_MERGED}/config.json'):
        print(f'\n  合并 Group C DPO LoRA → {G_C_MERGED}...')
        os.makedirs(G_C_MERGED, exist_ok=True)
        subprocess.run([
            'python3', 'scripts/merge_lora.py',
            '--adapter_path', G_C_DPO,
            '--base_model', SFT_BASE,
            '--output_path', G_C_MERGED,
        ], check=True)
        print(f'  ✅ Group C 合并完成: {G_C_MERGED}')

    # NF4 检测
    if os.path.isfile(f'{G_C_MERGED}/config.json'):
        G_C_MERGED = ensure_fp16_merged(G_C_MERGED, G_C_DPO, 'Group C DPO')
        print(f'  ✅ Group C 模型就绪: {G_C_MERGED}')
    else:
        print(f'  ❌ Group C 模型不存在（训练可能失败）')

    print('\nE6 完成')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E7: Group C 评测（GSM8K + MATH-500，n=200）
# 模型：outputs/group_c/merged（DoRA + 五段课程 SFT + Teacher DPO）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

# 确保 G_C_MERGED 变量存在
if 'G_C_MERGED' not in dir() or not G_C_MERGED:
    G_C_MERGED = 'outputs/group_c/merged'

if not os.path.isfile(f'{G_C_MERGED}/config.json'):
    print('⚠️ Group C DPO 模型不存在，跳过 E7')
    print(f'  检查: {G_C_MERGED}/config.json')
else:
    print('\n' + '='*60)
    print(f'  E7: Group C DPO 评测 (模型: {G_C_MERGED})')
    print('='*60)

    # GSM8K
    gsm_gc_out = 'logs/eval_supplement_group_c_dpo_gsm8k.json'
    if not os.path.isfile(gsm_gc_out):
        run_eval([
            'python3', 'eval/gsm8k_eval.py',
            '--model_path', G_C_MERGED,
            '--max_samples', EVAL_N,
            '--output', gsm_gc_out,
        ] + BIT, 'Group C DPO GSM8K')
    print_result(gsm_gc_out, 'Group C DPO GSM8K')

    # MATH-500
    math_gc_out = 'logs/eval_supplement_group_c_dpo_math.json'
    if not os.path.isfile(math_gc_out):
        run_eval([
            'python3', 'eval/math_eval.py',
            '--model_path', G_C_MERGED,
            '--max_samples', EVAL_N,
            '--output', math_gc_out,
        ] + BIT, 'Group C DPO MATH')
    print_result(math_gc_out, 'Group C DPO MATH')

    print('\nE7 完成')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E4: 汇总所有结果
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

def load_acc(path):
    if not os.path.isfile(path):
        return None, 0
    d = json.load(open(path))
    acc = d.get('accuracy', d.get('macro_avg_accuracy'))
    total = d.get('total', 0)
    return acc, total

# 本次补充评测
supplement = [
    ('1.5B Baseline', 'logs/eval_supplement_1.5b_gsm8k.json', 'logs/eval_supplement_1.5b_math.json'),
    ('7B Baseline', 'logs/eval_supplement_7b_gsm8k.json', 'logs/eval_supplement_7b_math.json'),
    ('Group A DPO', 'logs/eval_supplement_group_a_dpo_gsm8k.json', 'logs/eval_supplement_group_a_dpo_math.json'),
    ('Group C DPO', 'logs/eval_supplement_group_c_dpo_gsm8k.json', 'logs/eval_supplement_group_c_dpo_math.json'),
]

# 之前已有结果（logs 2/）
existing = [
    ('Group A SFT', 'logs 2/group_a_sft_gsm8k.json', 'logs 2/group_a_sft_math.json'),
    ('Group B SFT', 'logs 2/gsm8k_sft.json', 'logs 2/math_sft.json'),
    ('Group B DPO', 'logs 2/gsm8k_result.json', 'logs 2/math_result.json'),
    ('Group D', 'logs 2/group_d_gsm8k.json', 'logs 2/group_d_math.json'),
]

print('=' * 70)
print(f'{"模型":20s} {"GSM8K":>10s} {"MATH":>10s} {"n":>6s}')
print('-' * 70)

all_rows = []
for label, gsm_path, math_path in supplement + existing:
    gsm_acc, gsm_n = load_acc(gsm_path)
    math_acc, math_n = load_acc(math_path)
    gsm_s = f'{gsm_acc:.1%}' if gsm_acc is not None else '—'
    math_s = f'{math_acc:.1%}' if math_acc is not None else '—'
    n_s = str(gsm_n or math_n or '—')
    print(f'{label:20s} {gsm_s:>10s} {math_s:>10s} {n_s:>6s}')
    all_rows.append({'model': label, 'gsm8k': gsm_acc, 'math': math_acc, 'n': gsm_n or math_n})

print('=' * 70)

# 保存汇总
os.makedirs('results/ablation', exist_ok=True)
with open('results/ablation/supplement_summary.json', 'w') as f:
    json.dump(all_rows, f, ensure_ascii=False, indent=2)
print(f'\n汇总已保存: results/ablation/supplement_summary.json')
print('\n全部完成！')
